# Data Merging

Create three master csv files.
1. `master_student_program.csv`: One row = one student in one program. Useful for impact analysis, completion analysis, learning gain, satisfaction, dropout risk.
2. `daily_master.csv`: Useful for attendance trends, daily engagements, cohort behaviour over time.
3. `program_metrics.csv`: Impact reporting table.

## Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np

CLEAN_DIR = "../data/cleaned"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## Load Cleaned CSV Files

In [2]:
programs = pd.read_csv(f"{CLEAN_DIR}/programs_clean.csv")
users = pd.read_csv(f"{CLEAN_DIR}/users_clean.csv")
applications = pd.read_csv(f"{CLEAN_DIR}/applications_clean.csv")
pods = pd.read_csv(f"{CLEAN_DIR}/pods_clean.csv")
project_groups = pd.read_csv(f"{CLEAN_DIR}/project_groups_clean.csv")
ta_assignments = pd.read_csv(f"{CLEAN_DIR}/ta_assignments_clean.csv")
daily_engagement = pd.read_csv(f"{CLEAN_DIR}/daily_engagement_clean.csv")
assessments = pd.read_csv(f"{CLEAN_DIR}/assessments_clean.csv")
surveys = pd.read_csv(f"{CLEAN_DIR}/surveys_clean.csv")
dropouts = pd.read_csv(f"{CLEAN_DIR}/dropouts_clean.csv")
support_requests = pd.read_csv(f"{CLEAN_DIR}/support_requests_clean.csv")
ta_feedback = pd.read_csv(f"{CLEAN_DIR}/ta_feedback_clean.csv")
pod_checkins = pd.read_csv(f"{CLEAN_DIR}/pod_checkins_clean.csv")

## PART A — Build student-level master table

In [3]:
# Keep only accepted student-program enrollments
student_apps = applications[
    (applications["selected"] == 1) &
    (applications["final_status"] == "accepted")
].copy()

student_apps.shape

(372, 13)

In [4]:
# Merge with users
master = student_apps.merge(
    users,
    on="user_id",
    how="left"
)

master.shape
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5


In [5]:
# Merge with program metadata

master = master.merge(
    programs,
    on="program_id",
    how="left",
    suffixes=("", "_program")
)

master.shape
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5


## PART B — Add engagement features

In [ ]:
engagement_agg = daily_engagement.groupby(["user_id", "program_id"]).agg(
    total_days=("day", "nunique"),
    attended_curriculum_days=("attended_curriculum", "sum"),
    attended_project_days=("attended_project", "sum"),
    total_curriculum_hours=("curriculum_hours_attended", "sum"),
    total_project_hours=("project_hours_attended", "sum"),
    total_zoom_minutes=("zoom_minutes", "sum"),
    assignments_completed_total=("assignments_completed", "sum"),
    forum_messages_total=("forum_messages", "sum"),
    help_requests_total=("help_requests", "sum")
).reset_index()
engagement_agg.head()

,user_id,program_id,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total
0,U00002,P002,20,16,16,66.23,39.53,3792,16,100,27
1,U00007,P004,20,17,17,61.21,36.22,3653,17,82,18
2,U00008,P007,20,13,13,67.41,39.24,3364,13,108,31
3,U00014,P006,20,15,15,62.47,41.65,4111,15,106,41
4,U00016,P003,20,16,16,66.95,40.89,3111,16,116,24


In [7]:
engagement_agg["attendance_rate_curriculum"] = (
    engagement_agg["attended_curriculum_days"] / engagement_agg["total_days"]
)
engagement_agg["attendance_rate_project"] = (
    engagement_agg["attended_project_days"] / engagement_agg["total_days"]
)
engagement_agg["avg_zoom_minutes_per_day"] = (
    engagement_agg["total_zoom_minutes"] / engagement_agg["total_days"]
)
engagement_agg["avg_forum_messages_per_day"] = (
    engagement_agg["forum_messages_total"] / engagement_agg["total_days"]
)
engagement_agg["avg_help_requests_per_day"] = (
    engagement_agg["help_requests_total"] / engagement_agg["total_days"]
)
engagement_agg.head()

,user_id,program_id,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day
0,U00002,P002,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35
1,U00007,P004,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90
2,U00008,P007,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55
3,U00014,P006,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05
4,U00016,P003,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20


In [8]:
# Merge engagement into master
master = master.merge(
    engagement_agg,
    on=["user_id", "program_id"],
    how="left"
)

master.shape
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20


## PART C — Add learning outcomes

In [9]:
assessment_pivot = assessments.pivot_table(
    index=["user_id", "program_id"],
    columns="type",
    values="score",
    aggfunc="mean"
).reset_index()

assessment_pivot.columns.name = None
assessment_pivot.head()

,user_id,program_id,post,pre
0,U00002,P002,66.0,48.0
1,U00007,P004,52.0,47.0
2,U00008,P007,96.0,66.0
3,U00014,P006,61.0,53.0
4,U00016,P003,81.0,63.0


In [10]:
# Rename safely if needed
for col in ["pre", "mid", "post"]:
    if col not in assessment_pivot.columns:
        assessment_pivot[col] = np.nan

assessment_pivot = assessment_pivot.rename(columns={
    "pre": "pre_score",
    "mid": "mid_score",
    "post": "post_score"
})

assessment_pivot.head()

,user_id,program_id,post_score,pre_score,mid_score
0,U00002,P002,66.0,48.0,NaN
1,U00007,P004,52.0,47.0,NaN
2,U00008,P007,96.0,66.0,NaN
3,U00014,P006,61.0,53.0,NaN
4,U00016,P003,81.0,63.0,NaN


In [11]:
# Add learning gain
assessment_pivot["learning_gain"] = (
    assessment_pivot["post_score"] - assessment_pivot["pre_score"]
)

assessment_pivot.head()

,user_id,program_id,post_score,pre_score,mid_score,learning_gain
0,U00002,P002,66.0,48.0,NaN,18.0
1,U00007,P004,52.0,47.0,NaN,5.0
2,U00008,P007,96.0,66.0,NaN,30.0
3,U00014,P006,61.0,53.0,NaN,8.0
4,U00016,P003,81.0,63.0,NaN,18.0


In [12]:
# Merge assessments into master
master = master.merge(
    assessment_pivot,
    on=["user_id", "program_id"],
    how="left"
)

master.shape
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day,post_score,pre_score,mid_score,learning_gain
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35,66.0,48.0,NaN,18.0
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90,52.0,47.0,NaN,5.0
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55,96.0,66.0,NaN,30.0
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05,61.0,53.0,NaN,8.0
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20,81.0,63.0,NaN,18.0


## PART D — Add survey outcomes

In [13]:
survey_agg = surveys.groupby(["user_id", "program_id"]).agg(
    avg_satisfaction=("satisfaction", "mean"),
    avg_difficulty=("difficulty", "mean"),
    avg_content_quality=("content_quality", "mean"),
    avg_ta_support=("ta_support", "mean"),
    survey_response_count=("survey_id", "count")
).reset_index()

survey_agg.head()

,user_id,program_id,avg_satisfaction,avg_difficulty,avg_content_quality,avg_ta_support,survey_response_count
0,U00002,P002,4.0,4.0,5.0,4.0,1
1,U00007,P004,5.0,2.0,4.0,3.0,1
2,U00008,P007,5.0,3.0,4.0,3.0,1
3,U00014,P006,3.0,2.0,4.0,4.0,1
4,U00016,P003,3.0,2.0,4.0,4.0,1


In [14]:
# Merge survey data into master
master = master.merge(
    survey_agg,
    on=["user_id", "program_id"],
    how="left"
)

master.shape
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day,post_score,pre_score,mid_score,learning_gain,avg_satisfaction,avg_difficulty,avg_content_quality,avg_ta_support,survey_response_count
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35,66.0,48.0,NaN,18.0,4.0,4.0,5.0,4.0,1
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90,52.0,47.0,NaN,5.0,5.0,2.0,4.0,3.0,1
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55,96.0,66.0,NaN,30.0,5.0,3.0,4.0,3.0,1
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05,61.0,53.0,NaN,8.0,3.0,2.0,4.0,4.0,1
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20,81.0,63.0,NaN,18.0,3.0,2.0,4.0,4.0,1


## PART E — Add dropout info

In [15]:
# Merge dropout status
dropouts["dropped_out"] = 1

master = master.merge(
    dropouts[["user_id", "program_id", "dropout_day", "reason", "dropped_out"]],
    on=["user_id", "program_id"],
    how="left"
)

master["dropped_out"] = master["dropped_out"].fillna(0).astype(int)
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day,post_score,pre_score,mid_score,learning_gain,avg_satisfaction,avg_difficulty,avg_content_quality,avg_ta_support,survey_response_count,dropout_day,reason,dropped_out
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35,66.0,48.0,NaN,18.0,4.0,4.0,5.0,4.0,1,NaN,NaN,0
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90,52.0,47.0,NaN,5.0,5.0,2.0,4.0,3.0,1,NaN,NaN,0
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55,96.0,66.0,NaN,30.0,5.0,3.0,4.0,3.0,1,NaN,NaN,0
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05,61.0,53.0,NaN,8.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20,81.0,63.0,NaN,18.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0


## PART F — Add support burden

In [16]:
# Aggregate support requests 
support_agg = support_requests.groupby(["user_id", "program_id"]).agg(
    support_ticket_count=("ticket_id", "count"),
    avg_support_response_time=("response_time_hours", "mean"),
    avg_support_satisfaction=("satisfaction", "mean")
).reset_index()

support_agg.head()

,user_id,program_id,support_ticket_count,avg_support_response_time,avg_support_satisfaction
0,U00003,P008,1,5.620,3.0
1,U00018,P001,2,7.945,4.5
2,U00021,P003,1,12.640,5.0
3,U00026,P002,1,6.030,4.0
4,U00028,P005,1,20.950,5.0


In [17]:
# Merge support into master
master = master.merge(
    support_agg,
    on=["user_id", "program_id"],
    how="left"
)

master.shape
master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day,post_score,pre_score,mid_score,learning_gain,avg_satisfaction,avg_difficulty,avg_content_quality,avg_ta_support,survey_response_count,dropout_day,reason,dropped_out,support_ticket_count,avg_support_response_time,avg_support_satisfaction
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35,66.0,48.0,NaN,18.0,4.0,4.0,5.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90,52.0,47.0,NaN,5.0,5.0,2.0,4.0,3.0,1,NaN,NaN,0,NaN,NaN,NaN
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55,96.0,66.0,NaN,30.0,5.0,3.0,4.0,3.0,1,NaN,NaN,0,NaN,NaN,NaN
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05,61.0,53.0,NaN,8.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20,81.0,63.0,NaN,18.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN


## PART G — Create final derived student-level features

In [18]:
# Final Feature Engineering
# Completion rule
master["completion_status"] = np.where(
    (master["attendance_rate_curriculum"] >= 0.7) &
    (master["attendance_rate_project"] >= 0.7),
    1, 0
)

# Engagement score (simple weighted score)
master["engagement_score"] = (
    0.30 * master["attendance_rate_curriculum"].fillna(0) +
    0.25 * master["attendance_rate_project"].fillna(0) +
    0.20 * (master["assignments_completed_total"].fillna(0) / master["total_days"].replace(0, np.nan)) +
    0.15 * (master["forum_messages_total"].fillna(0) / master["total_days"].replace(0, np.nan)) +
    0.10 * (1 - (master["help_requests_total"].fillna(0) / master["total_days"].replace(0, np.nan)))
)

master["engagement_score"] = master["engagement_score"].fillna(0).clip(0, 1)

# Survey response rate proxy
master["survey_response_rate"] = master["survey_response_count"].fillna(0) / 3

master.head()

,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day,post_score,pre_score,mid_score,learning_gain,avg_satisfaction,avg_difficulty,avg_content_quality,avg_ta_support,survey_response_count,dropout_day,reason,dropped_out,support_ticket_count,avg_support_response_time,avg_support_satisfaction,completion_status,engagement_score,survey_response_rate
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35,66.0,48.0,NaN,18.0,4.0,4.0,5.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90,52.0,47.0,NaN,5.0,5.0,2.0,4.0,3.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55,96.0,66.0,NaN,30.0,5.0,3.0,4.0,3.0,1,NaN,NaN,0,NaN,NaN,NaN,0,1.0,0.333333
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05,61.0,53.0,NaN,8.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20,81.0,63.0,NaN,18.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333


## PART H — Create enriched daily-level table

In [19]:
# Enrich daily engagement with user + program info
daily_master = daily_engagement.merge(
    users,
    on="user_id",
    how="left"
).merge(
    programs,
    on="program_id",
    how="left"
)

daily_master.shape
daily_master.head()

,user_id,program_id,day,attended_curriculum,attended_project,curriculum_hours_attended,project_hours_attended,zoom_minutes,assignments_completed,forum_messages,help_requests,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots
0,U00002,P002,1,1,1,3.85,1.16,133,1,1,1,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5
1,U00002,P002,2,1,1,2.86,1.74,278,1,8,2,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5
2,U00002,P002,3,1,1,3.97,2.50,162,1,9,0,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5
3,U00002,P002,4,0,0,2.74,2.25,199,0,1,3,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5
4,U00002,P002,5,0,0,2.18,2.66,299,0,4,0,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5


## Create Program Metrics

In [20]:
program_metrics = master.groupby(["program_id", "year"]).agg(
    completion_rate=("completion_status", "mean"),
    avg_learning_gain=("learning_gain", "mean"),
    avg_satisfaction=("avg_satisfaction", "mean"),
    dropout_rate=("dropped_out", "mean"),
    engagement_score=("engagement_score", "mean")
).reset_index()

program_metrics.head()

,program_id,year,completion_rate,avg_learning_gain,avg_satisfaction,dropout_rate,engagement_score
0,P001,2025,0.926829,17.292683,4.048780,0.097561,1.0
1,P002,2025,0.941176,16.352941,3.843137,0.098039,1.0
2,P003,2025,0.891892,13.756757,4.000000,0.054054,1.0
3,P004,2025,0.833333,16.592593,3.870370,0.129630,1.0
4,P005,2026,0.882353,18.176471,4.000000,0.117647,1.0


In [21]:
rate_cols = ["completion_rate", "dropout_rate", "engagement_score"]

for col in rate_cols:
    program_metrics[col] = (program_metrics[col] * 100).round(2)

program_metrics["avg_learning_gain"] = program_metrics["avg_learning_gain"].round(2)
program_metrics["avg_satisfaction"] = program_metrics["avg_satisfaction"].round(2)

program_metrics.head()

,program_id,year,completion_rate,avg_learning_gain,avg_satisfaction,dropout_rate,engagement_score
0,P001,2025,92.68,17.29,4.05,9.76,100.0
1,P002,2025,94.12,16.35,3.84,9.80,100.0
2,P003,2025,89.19,13.76,4.00,5.41,100.0
3,P004,2025,83.33,16.59,3.87,12.96,100.0
4,P005,2026,88.24,18.18,4.00,11.76,100.0


## Save Outputs

In [22]:
master.to_csv(f"{PROCESSED_DIR}/master_student_program.csv", index=False)
daily_master.to_csv(f"{PROCESSED_DIR}/daily_master.csv", index=False)
program_metrics.to_csv(f"{PROCESSED_DIR}/program_metrics.csv", index=False)

print("✅ Saved:")
print("- master_student_program.csv")
print("- daily_master.csv")
print("- program_metrics.csv")

✅ Saved:
- master_student_program.csv
- daily_master.csv
- program_metrics.csv


## Quick validation

In [23]:
print("Master shape:", master.shape)
print("Daily master shape:", daily_master.shape)
print("Program metrics shape:", program_metrics.shape)

display(master.head())
display(program_metrics.head())

Master shape: (372, 69)
Daily master shape: (7440, 35)
Program metrics shape: (8, 7)


,application_id,user_id,program_id,application_date,motivation_score,cv_score,technical_score,diversity_score,preferred_program_track,preferred_timeslot,selected,waitlisted,final_status,role,country,timezone,gender,education_level,field_of_study,years_experience,prior_neuromatch,preferred_language,interest_area,availability_slot,program_name,year,start_date,end_date,num_students,num_tas,region_focus,curriculum_hours_per_day,project_hours_per_day,total_hours_per_day,days_per_week,delivery_mode,num_timeslots,total_days,attended_curriculum_days,attended_project_days,total_curriculum_hours,total_project_hours,total_zoom_minutes,assignments_completed_total,forum_messages_total,help_requests_total,attendance_rate_curriculum,attendance_rate_project,avg_zoom_minutes_per_day,avg_forum_messages_per_day,avg_help_requests_per_day,post_score,pre_score,mid_score,learning_gain,avg_satisfaction,avg_difficulty,avg_content_quality,avg_ta_support,survey_response_count,dropout_day,reason,dropped_out,support_ticket_count,avg_support_response_time,avg_support_satisfaction,completion_status,engagement_score,survey_response_rate
0,A000002,U00002,P002,2026-03-01,67.661960,60.104138,68.089247,88,Research,Slot 4,1,0,accepted,student,Canada,UTC-5,Male,Masters,Neuro,2,0,English,CV,Slot 4,Deep Learning,2025,2025-06-22,2025-07-24,206,23,global,4.5,3,8,5,virtual,5,20,16,16,66.23,39.53,3792,16,100,27,0.80,0.80,189.60,5.0,1.35,66.0,48.0,NaN,18.0,4.0,4.0,5.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333
1,A000006,U00007,P004,2026-03-01,84.445828,44.577177,100.000000,54,Research,Slot 3,1,0,accepted,student,USA,UTC-5,Female,Masters,Climate,8,0,French,CV,Slot 3,NeuroAI,2025,2025-06-22,2025-07-24,235,18,global,4.5,3,8,5,virtual,5,20,17,17,61.21,36.22,3653,17,82,18,0.85,0.85,182.65,4.1,0.90,52.0,47.0,NaN,5.0,5.0,2.0,4.0,3.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333
2,A000007,U00008,P007,2026-03-01,50.724878,51.085213,99.826941,86,Research,Slot 2,1,0,accepted,student,UK,UTC+0,Male,PhD,Neuro,9,0,English,Climate,Slot 2,Computational Tools for Climate Science,2026,2026-06-22,2026-07-24,208,15,global,4.5,3,8,5,virtual,5,20,13,13,67.41,39.24,3364,13,108,31,0.65,0.65,168.20,5.4,1.55,96.0,66.0,NaN,30.0,5.0,3.0,4.0,3.0,1,NaN,NaN,0,NaN,NaN,NaN,0,1.0,0.333333
3,A000011,U00014,P006,2026-03-01,55.956026,54.914295,100.000000,49,Research,Slot 2,1,0,accepted,student,Brazil,UTC-3,Non-binary,Masters,CS,6,0,English,Neural Coding,Slot 2,Deep Learning,2026,2026-06-22,2026-07-24,351,28,global,4.5,3,8,5,virtual,5,20,15,15,62.47,41.65,4111,15,106,41,0.75,0.75,205.55,5.3,2.05,61.0,53.0,NaN,8.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333
4,A000013,U00016,P003,2026-03-01,55.969240,69.915533,78.250660,45,Research,Slot 5,1,0,accepted,student,Japan,UTC+9,Non-binary,Masters,CS,7,0,English,Time Series,Slot 5,Computational Tools for Climate Science,2025,2025-06-22,2025-07-24,262,22,global,4.5,3,8,5,virtual,5,20,16,16,66.95,40.89,3111,16,116,24,0.80,0.80,155.55,5.8,1.20,81.0,63.0,NaN,18.0,3.0,2.0,4.0,4.0,1,NaN,NaN,0,NaN,NaN,NaN,1,1.0,0.333333


,program_id,year,completion_rate,avg_learning_gain,avg_satisfaction,dropout_rate,engagement_score
0,P001,2025,92.68,17.29,4.05,9.76,100.0
1,P002,2025,94.12,16.35,3.84,9.80,100.0
2,P003,2025,89.19,13.76,4.00,5.41,100.0
3,P004,2025,83.33,16.59,3.87,12.96,100.0
4,P005,2026,88.24,18.18,4.00,11.76,100.0
